# 01 — Il riferimento: il TF-IDF per articolo delle Versioni 1–4 (tentativo 09)

Il notebook del tentativo 06, invariato, sul dataset del tentativo 09 (gruppi con anche il Parent Domain, 301 testate
valutate): è il termine di confronto per i modelli nuovi. Si lancia con `C_GRIGLIA = "0.1,1,10,100"` e `NOME = "feature_tuned"`.

## Il notebook originale (Versione 2)

Sul dataset finale (`1-data-cleaning/03_dataset_finale`): solo il **testo degli articoli**.

**Versione 2.** Nel training entrano anche le testate di solo training (tutte le testate dei gruppi di training,
con almeno 5 articoli), ognuna con il suo **peso** (un gruppo vale al più 5 testate); le classi si bilanciano
sui pesi. Il test e le metriche sono solo sulle testate **valutate**, le stesse della prima versione. Con
`SOLO_VALUTAZIONE = True` si addestra solo sulle testate valutate, come nella prima versione (il controllo).

Fold **per gruppo** (la valutazione principale: tutte le testate di un gruppo, cioè rete o stesso owner, nello
stesso fold) e **per testata** (confronto: i gruppi possono essere spezzati). Un modello per fold: vocabolario e
IDF si stimano **solo sul training** del fold, il test passa solo da `transform`. Iperparametri fissi.

Con `TAGLIO_TOKEN` il testo si taglia ai primi N token del tokenizer di ModernBERT: lo stesso testo che vede il
transformer con `MAX_TOKEN` = N.

Accanto al classificatore due baseline senza testo: la **classe di maggioranza** del training e una **scelta a
caso** con le proporzioni del training.

Si salva **tutto**, per poter ricalcolare qualsiasi metrica senza riaddestrare:

- per ogni articolo del test: `article_id`, `domain`, `fold`, `y_true`, i **logit** `logit_l1..logit_l5`, le
  **probabilità** `p_l1..p_l5`, `y_pred`;
- `metriche.json`: per articolo, per testata col voto e per testata con la media delle probabilità —
  accuratezza, F1 macro e pesata, precisione/recall/F1 per livello, matrice di confusione, tolerant 0.5 e 1,
  errore medio in livelli, binario (soglia $l_2$/$l_3$) con TP, TN, FP, FN, precisione, recall, F1.

File: `Results/2-classificazione/<NOME>_<schema>_articoli.csv`, `<NOME>_<schema>_metriche.json`,
`<NOME>_metriche.csv` (tabella riassuntiva), `<NOME>_FATTO` quando tutto è finito.

**Sul server:** `cd ~/John/WebScience2027/Tentativi/06_Versione2_PiuPuliziaPiuTraining/2-classificazione && papermill -k johnvenv 01_classificatore_feature.ipynb ../Results/2-classificazione/esecuzioni/01_classificatore_feature.ipynb`
(di solito lo lancia `coda.py`).

In [1]:
import json
import time

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold

from comune import CLASSI, NOME_SCHEMA, SCHEMI, SEED, carica, cartella_risultati, letture, metriche, per_testata, pesi_bilanciati, registratore

In [2]:
NOME = "feature"
TAGLIO_TOKEN = 0           # 0 = testo intero; N = primi N token (tokenizer di ModernBERT)
C_GRIGLIA = ""             # "" = C fisso a 1; "0.1,1,10,100" = C scelto con una validazione interna per gruppi (solo training)
SCHEMI_DA_FARE = SCHEMI
SOLO_VALUTAZIONE = False   # True: training solo sulle testate valutate (come la prima versione)
N_JOBS = 5                 # fold in parallelo
PROVA = False              # True: 40 testate, risultati in Results/2-classificazione/prova/

In [3]:
# Parameters
SCHEMI_DA_FARE = ["fold_gruppo"]
NOME = "feature_tuned"
C_GRIGLIA = "0.1,1,10,100"
N_JOBS = 5


In [4]:
OUT = cartella_risultati(PROVA)
registra = registratore(OUT / f"avanzamento_{NOME}.log")
art = carica(PROVA, SOLO_VALUTAZIONE)
if TAGLIO_TOKEN:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
    enc = tok(art.text.tolist(), truncation=True, max_length=TAGLIO_TOKEN, add_special_tokens=False, return_offsets_mapping=True)
    art["text"] = [t[:o[-1][1]] if o else t for t, o in zip(art.text, enc["offset_mapping"])]
registra(f"=== {NOME}: {len(art)} articoli, {art.domain.nunique()} testate ({art[art.valutazione].domain.nunique()} valutate), "
         f"taglio {TAGLIO_TOKEN or 'nessuno'}, solo valutate {SOLO_VALUTAZIONE} ===")

2026-10-22 14:55:14  === feature_tuned: 75394 articoli, 863 testate (301 valutate), taglio nessuno, solo valutate False ===


## Addestramento

Per ogni fold: `fit_transform` sul training, `transform` sul test. I logit sono la `decision_function` della
regressione logistica.

In [5]:
def nuovo_tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.9, sublinear_tf=True,
                           max_features=100_000, strip_accents="unicode")


def scegli_C(tr):
    """C con la F1 macro per testata (voto) più alta in una validazione a 3 fold per gruppi, sul solo training."""
    griglia = [float(x) for x in C_GRIGLIA.split(",")]
    testate = tr.drop_duplicates("domain").reset_index(drop=True)
    punteggi = {c: [] for c in griglia}
    for itr, iva in StratifiedGroupKFold(3, shuffle=True, random_state=SEED).split(testate, testate.label, testate.gruppo):
        a = tr[tr.domain.isin(testate.domain[itr])]
        b = tr[tr.domain.isin(testate.domain[iva]) & tr.valutazione]          # si valuta come nel test: solo testate valutate
        v = nuovo_tfidf(); Xa, Xb = v.fit_transform(a.text), v.transform(b.text)
        pa = pesi_bilanciati(a.label, a.peso)
        for c in griglia:
            p = LogisticRegression(C=c, max_iter=1000, random_state=SEED).fit(Xa, a.label, sample_weight=pa).predict(Xb)
            t = per_testata(pd.DataFrame({"domain": b.domain.values, "y_true": b.label.values, "y_pred": p}))
            punteggi[c].append(f1_score(t.y_true, t.y_pred, average="macro", labels=range(5), zero_division=0))
    medie = {c: float(np.mean(s)) for c, s in punteggi.items()}
    return max(medie, key=medie.get), medie


def un_fold(schema, fold):
    t0 = time.time()
    tr, te = art[art[schema] != fold], art[(art[schema] == fold) & art.valutazione]
    C, medie = (1.0, {}) if not C_GRIGLIA else scegli_C(tr)
    if C_GRIGLIA:
        registra(f"{schema} fold {fold}: C scelto {C} (F1 interne {', '.join(f'{k}: {v:.3f}' for k, v in medie.items())})")
    v = nuovo_tfidf()
    clf = LogisticRegression(C=C, max_iter=1000, random_state=SEED)
    clf.fit(v.fit_transform(tr.text), tr.label, sample_weight=pesi_bilanciati(tr.label, tr.peso))   # vocabolario, IDF e modello: solo training
    X = v.transform(te.text)                                   # test: solo transform
    registra(f"{schema} fold {fold}: fatto in {time.time() - t0:.0f} s ({len(tr)} training, {len(te)} test)")
    return te.index.to_numpy(), clf.classes_, clf.decision_function(X), clf.predict_proba(X)


def addestra(schema):
    file_out = OUT / f"{NOME}_{schema}_articoli.csv"
    if file_out.exists():
        registra(f"{schema}: già fatto, carico {file_out.name}")
        return pd.read_csv(file_out)
    logit, proba = np.full((len(art), 5), np.nan), np.zeros((len(art), 5))
    for idx, classi, z, p in Parallel(n_jobs=N_JOBS)(delayed(un_fold)(schema, f) for f in sorted(art[schema].unique())):
        logit[idx[:, None], classi[None, :]] = z
        proba[idx[:, None], classi[None, :]] = p
    pred = pd.DataFrame({"article_id": art.article_id, "domain": art.domain, "fold": art[schema], "y_true": art.label})
    for k in range(5):
        pred[f"logit_l{k + 1}"] = logit[:, k]
    for k, c in enumerate(CLASSI):
        pred[c] = proba[:, k]
    pred["y_pred"] = proba.argmax(1)
    pred = pred[art.valutazione.to_numpy()].reset_index(drop=True)      # solo le testate valutate
    pred.to_csv(file_out, index=False)
    json.dump(metriche(pred), open(OUT / f"{NOME}_{schema}_metriche.json", "w"), indent=1)
    return pred


pred = {s: addestra(s) for s in SCHEMI_DA_FARE}

2026-10-22 15:09:12  fold_gruppo fold 3: C scelto 0.1 (F1 interne 0.1: 0.331, 1.0: 0.316, 10.0: 0.300, 100.0: 0.295)


2026-10-22 15:09:55  fold_gruppo fold 2: C scelto 1.0 (F1 interne 0.1: 0.372, 1.0: 0.404, 10.0: 0.392, 100.0: 0.396)


2026-10-22 15:10:03  fold_gruppo fold 4: C scelto 0.1 (F1 interne 0.1: 0.329, 1.0: 0.314, 10.0: 0.304, 100.0: 0.287)


2026-10-22 15:10:09  fold_gruppo fold 1: C scelto 1.0 (F1 interne 0.1: 0.268, 1.0: 0.269, 10.0: 0.234, 100.0: 0.251)


2026-10-22 15:11:25  fold_gruppo fold 3: fatto in 966 s (58419 training, 7654 test)


2026-10-22 15:11:43  fold_gruppo fold 5: C scelto 0.1 (F1 interne 0.1: 0.287, 1.0: 0.261, 10.0: 0.250, 100.0: 0.263)


2026-10-22 15:12:33  fold_gruppo fold 4: fatto in 1033 s (62350 training, 9720 test)


2026-10-22 15:13:00  fold_gruppo fold 2: fatto in 1062 s (55613 training, 9304 test)


2026-10-22 15:13:10  fold_gruppo fold 1: fatto in 1073 s (58088 training, 8723 test)


2026-10-22 15:14:28  fold_gruppo fold 5: fatto in 1147 s (67106 training, 7933 test)


## Baseline senza testo

Per ogni fold, dalle testate valutate di training (come nella prima versione): la classe più frequente, e una classe estratta a caso con le
proporzioni del training (una per testata, seed fisso).

In [6]:
def baseline(schema, modo):
    rng = np.random.default_rng(SEED)
    y = np.full(len(art), -1)
    for fold in sorted(art[schema].unique()):
        tr, te = art[(art[schema] != fold) & art.valutazione], art[(art[schema] == fold) & art.valutazione]
        priori = tr.drop_duplicates("domain").label.value_counts(normalize=True).sort_index()
        if modo == "maggioranza":
            y[te.index] = priori.idxmax()
        else:
            scelte = dict(zip(te.domain.unique(), rng.choice(priori.index, te.domain.nunique(), p=priori.values)))
            y[te.index] = te.domain.map(scelte)
    b = pd.DataFrame({"domain": art.domain, "y_true": art.label, "y_pred": y})
    return b[art.valutazione.to_numpy()].reset_index(drop=True)


for s in SCHEMI_DA_FARE:
    for modo in ["maggioranza", "a caso"]:
        json.dump(metriche(baseline(s, modo)), open(OUT / f"baseline_{modo.replace(' ', '_')}_{s}_metriche.json", "w"), indent=1)

## Risultati

Per articolo e per testata (voto di maggioranza): accuratezza a 5 classi, F1 macro, binario e tolerant.

In [7]:
righe = []
for s in SCHEMI_DA_FARE:
    righe.append({"modello": f"TF-IDF + LR ({NOME})", "fold": NOME_SCHEMA[s], **letture(pred[s])})
    for modo in ["maggioranza", "a caso"]:
        righe.append({"modello": f"baseline: {modo}", "fold": NOME_SCHEMA[s], **letture(baseline(s, modo))})
ris = pd.DataFrame(righe).set_index(["fold", "modello"])
ris.to_csv(OUT / f"{NOME}_metriche.csv")
ris[[c for c in ris.columns if c.startswith("testata")] + ["articolo 5cl", "articolo bin"]]

testata 5cl  testata F1  testata bin  \
fold       modello                                                             
per gruppo TF-IDF + LR (feature_tuned)        0.369       0.340        0.684   
           baseline: maggioranza              0.316       0.096        0.561   
           baseline: a caso                   0.216       0.176        0.485   

                                        testata tol0.5  testata tol1  \
fold       modello                                                     
per gruppo TF-IDF + LR (feature_tuned)           0.555         0.741   
           baseline: maggioranza                 0.439         0.561   
           baseline: a caso                      0.369         0.522   

                                        articolo 5cl  articolo bin  
fold       modello                                                  
per gruppo TF-IDF + LR (feature_tuned)         0.297         0.650  
           baseline: maggioranza               0.310         0.558  
           baseline: a caso                    0.212         0.471

In [8]:
t = per_testata(pred[SCHEMI_DA_FARE[0]])
LIV = ["l1", "l2", "l3", "l4", "l5"]
display(pd.crosstab(t.y_true.map(dict(enumerate(LIV))), t.y_pred.map(dict(enumerate(LIV))), rownames=["vero"], colnames=["predetto"]))
(OUT / f"{NOME}_FATTO").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))

predetto,l1,l2,l3,l4,l5
vero,,,,,
l1,39,15,4,8,2
l2,12,28,9,8,7
l3,4,14,7,11,2
l4,10,18,20,29,18
l5,3,8,4,13,8


19